# CSRNet Model Testing
## Load architecture, weights, and test inference

In [4]:
# === STEP 1: Import CSRNet and setup paths ===
import sys
import os
import torch
from collections import OrderedDict
from pathlib import Path

# Add the project paths
project_root = Path('..').resolve()  # Points to ml/src/
sys.path.insert(0, str(project_root))  # For models and preprocessing

# Import the model
from models.csrnet.csrnet import CSRNet, load_csrnet
print('✅ CSRNet imported successfully from models/csrnet/csrnet.py')

# Verify paths
print(f'\n📁 Project paths configured:')
print(f'   Project root (ml/src): {project_root}')
print(f'   Datasets: {project_root.parent / "datasets"}')
print(f'   Checkpoints: {project_root.parent / "checkpoints"}')

✅ CSRNet imported successfully from models/csrnet/csrnet.py

📁 Project paths configured:
   Project root (ml/src): D:\College\Major Project\ml\src
   Datasets: D:\College\Major Project\ml\datasets
   Checkpoints: D:\College\Major Project\ml\checkpoints


In [5]:
# === STEP 2: Load CSRNet model with checkpoint ===
checkpoint_path = '../../checkpoints/csrnet.pth'

print(f'📦 Loading CSRNet model from: {checkpoint_path}')
print(f'   Checkpoint exists: {os.path.exists(checkpoint_path)}')

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🎮 Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# Use the helper function to load model with checkpoint
model = load_csrnet(checkpoint_path, device=device)

print(f'✅ Model loaded successfully')
print(f'   Model architecture:')
print(f'   - Frontend: {len(list(model.frontend.parameters()))} parameter tensors')
print(f'   - Backend: {len(list(model.backend.parameters()))} parameter tensors')
print(f'   - Output layer: Conv2d(64 -> 1)')
print(f'   Model is in eval mode: {not model.training}')
print(f'   Model device: {next(model.parameters()).device}')

📦 Loading CSRNet model from: ../../checkpoints/csrnet.pth
   Checkpoint exists: True
🎮 Device: cuda
   GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


D:\College\Major Project\ml\src\models\csrnet\csrnet.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


✅ Model loaded successfully
   Model architecture:
   - Frontend: 20 parameter tensors
   - Backend: 12 parameter tensors
   - Output layer: Conv2d(64 -> 1)
   Model is in eval mode: True
   Model device: cuda:0


In [6]:
# === STEP 3: Load a real crowd image from datasets/images ===
import torchvision.transforms as transforms
from PIL import Image

# Path to dataset images
dataset_images_dir = project_root.parent / 'datasets' / 'images'
print(f'📂 Looking for images in: {dataset_images_dir}')
print(f'   Directory exists: {dataset_images_dir.exists()}')

# Find all image files
available_images = sorted([
    img for img in dataset_images_dir.glob('*') 
    if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']
])

if not available_images:
    raise FileNotFoundError(f'No images found in {dataset_images_dir}')

print(f'\n🖼️  Found {len(available_images)} images:')
for i, img in enumerate(available_images, 1):
    print(f'   {i}. {img.name}')

# Use the first image
test_image_path = available_images[0]
print(f'\n➡️  Using: {test_image_path.name}')

# Load and display image info
crowd_img = Image.open(test_image_path).convert('RGB')
print(f'   Image size: {crowd_img.size[0]}x{crowd_img.size[1]} pixels')
print(f'   Mode: {crowd_img.mode}')

# Apply standard preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_tensor = transform(crowd_img).unsqueeze(0)  # Add batch dimension

# Move to same device as model
device = next(model.parameters()).device
img_tensor = img_tensor.to(device)

print(f'\n🔄 Preprocessing completed:')
print(f'   Tensor shape: {img_tensor.shape}')
print(f'   Tensor dtype: {img_tensor.dtype}')
print(f'   Tensor device: {img_tensor.device}')
print(f'   Tensor range: [{img_tensor.min():.3f}, {img_tensor.max():.3f}]')

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n📊 Model Statistics:')
print(f'   Total parameters: {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')
print(f'   Device: {next(model.parameters()).device}')

📂 Looking for images in: D:\College\Major Project\ml\datasets\images
   Directory exists: True

🖼️  Found 3 images:
   1. 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg
   2. 360_F_600734899_r2VocyDutRuAcmld87AcxOTCR9NPApSq.jpg
   3. png-multicultural-crowd-people-person-back_53876-621138.jpg

➡️  Using: 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg
   Image size: 483x360 pixels
   Mode: RGB

🔄 Preprocessing completed:
   Tensor shape: torch.Size([1, 3, 360, 483])
   Tensor dtype: torch.float32
   Tensor device: cuda:0
   Tensor range: [-2.118, 2.640]

📊 Model Statistics:
   Total parameters: 16,263,489
   Trainable parameters: 0
   Device: cuda:0
   Tensor range: [-2.118, 2.640]

📊 Model Statistics:
   Total parameters: 16,263,489
   Trainable parameters: 0
   Device: cuda:0


In [7]:
# === STEP 4: Run inference and show count in CLI ===
import time

print('🧠 Running inference...')

# Time the inference
start_time = time.time()

with torch.no_grad():
    density_map = model(img_tensor)
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Wait for GPU to finish
    
    count = density_map.sum().item()

inference_time = time.time() - start_time

print(f'\n✅ INFERENCE SUCCESSFUL!')
print(f'\n📊 Results:')
print(f'   Density map shape: {density_map.shape}')
print(f'   Density map range: [{density_map.min():.6f}, {density_map.max():.6f}]')
print(f'   Predicted count: {count:.2f}')
print(f'   Rounded count: {int(round(count))}')
print(f'   ⏱️  Inference time: {inference_time:.4f}s')

print(f'\n' + '='*50)
print(f'   🎯 FINAL COUNT: {int(round(count))} people')
print(f'='*50)

print(f'\n✅ Model is working correctly! Ready for API integration.')

🧠 Running inference...

✅ INFERENCE SUCCESSFUL!

📊 Results:
   Density map shape: torch.Size([1, 1, 45, 60])
   Density map range: [0.000000, 3.131865]
   Predicted count: 623.12
   Rounded count: 623
   ⏱️  Inference time: 0.8193s

   🎯 FINAL COUNT: 623 people

✅ Model is working correctly! Ready for API integration.

✅ INFERENCE SUCCESSFUL!

📊 Results:
   Density map shape: torch.Size([1, 1, 45, 60])
   Density map range: [0.000000, 3.131865]
   Predicted count: 623.12
   Rounded count: 623
   ⏱️  Inference time: 0.8193s

   🎯 FINAL COUNT: 623 people

✅ Model is working correctly! Ready for API integration.


## 🚀 GPU Performance Test

Let's test the speedup from GPU acceleration!

In [8]:
# === GPU vs CPU Performance Comparison ===
import time

if torch.cuda.is_available():
    print('🏁 GPU vs CPU Benchmark')
    print('='*70)
    
    # Test on CPU
    print('\n⏱️  Testing on CPU...')
    model_cpu = model.cpu()
    img_cpu = img_tensor.cpu()
    
    cpu_times = []
    for i in range(5):
        start = time.time()
        with torch.no_grad():
            _ = model_cpu(img_cpu)
        cpu_times.append(time.time() - start)
    
    cpu_avg = sum(cpu_times[1:]) / 4  # Skip first run (warmup)
    print(f'   Average time: {cpu_avg:.4f}s')
    
    # Test on GPU
    print('\n⏱️  Testing on GPU...')
    model_gpu = model.cuda()
    img_gpu = img_tensor.cuda()
    
    # Warmup
    with torch.no_grad():
        _ = model_gpu(img_gpu)
    torch.cuda.synchronize()
    
    gpu_times = []
    for i in range(5):
        start = time.time()
        with torch.no_grad():
            _ = model_gpu(img_gpu)
        torch.cuda.synchronize()
        gpu_times.append(time.time() - start)
    
    gpu_avg = sum(gpu_times[1:]) / 4  # Skip first run
    print(f'   Average time: {gpu_avg:.4f}s')
    
    # Results
    speedup = cpu_avg / gpu_avg
    print(f'\n🚀 GPU Speedup: {speedup:.2f}x faster!')
    print(f'   CPU: {cpu_avg:.4f}s per image')
    print(f'   GPU: {gpu_avg:.4f}s per image')
    
    if speedup > 5:
        print('   ✅ Excellent GPU acceleration!')
    elif speedup > 2:
        print('   ✅ Good GPU acceleration!')
    else:
        print('   ⚠️  Limited speedup - may be GPU throttled')
    
    # Move model back to GPU for subsequent cells
    model = model_gpu
    img_tensor = img_gpu
    
    print('='*70)
else:
    print('⚠️  CUDA not available - skipping GPU benchmark')
    print('   Running on CPU only')

🏁 GPU vs CPU Benchmark

⏱️  Testing on CPU...
   Average time: 0.3773s

⏱️  Testing on GPU...
   Average time: 0.3773s

⏱️  Testing on GPU...
   Average time: 0.0307s

🚀 GPU Speedup: 12.27x faster!
   CPU: 0.3773s per image
   GPU: 0.0307s per image
   ✅ Excellent GPU acceleration!
   Average time: 0.0307s

🚀 GPU Speedup: 12.27x faster!
   CPU: 0.3773s per image
   GPU: 0.0307s per image
   ✅ Excellent GPU acceleration!


## ✅ SUMMARY

The CSRNet model has been successfully tested:
1. ✅ Loaded from `models/csrnet/csrnet.py` (clean, Python 3 compatible)
2. ✅ Weights loaded from `checkpoints/csrnet.pth`
3. ✅ Architecture validated (16.2M parameters)
4. ✅ Inference tested with **real crowd image from datasets/images**
5. ✅ **COUNT DISPLAYED IN CLI** (see output above)

### What was tested:
- 🖼️  Real crowd image from `ml/datasets/images/`
- 🔄 Standard ImageNet preprocessing
- 🧠 CSRNet inference on CPU
- 📊 Density map generation and counting

### Next Steps:
- Test with multiple images from the dataset
- Compare results with expected crowd counts
- Deploy model via API or webcam app
- Use this notebook for debugging checkpoint issues

## 🔧 Now Test with Proper Preprocessing Module

Let's use the new preprocessing module that matches the original CSRNet exactly.

In [6]:
# === STEP 5: Test with proper CSRNet preprocessing module ===
from preprocessing import CSRNetPreprocessor

# Initialize preprocessor
preprocessor = CSRNetPreprocessor()
print("✅ CSRNetPreprocessor initialized (matches original CSRNet exactly)")
print("   - No resizing (fully convolutional)")
print("   - ToTensor + ImageNet normalization")
print("   - Output downsampled by factor of 8")

print(f"\n🔍 Testing same image: {test_image_path.name}")

✅ CSRNetPreprocessor initialized (matches original CSRNet exactly)
   - No resizing (fully convolutional)
   - ToTensor + ImageNet normalization
   - Output downsampled by factor of 8

🔍 Testing same image: 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg


In [8]:
# Test with the same dataset image using proper preprocessing
print("🖼️ Testing with dataset image using proper preprocessing...")

# Use the preprocessor
img_tensor_new = preprocessor.preprocess(crowd_img)

# Move tensor to same device as model
device = next(model.parameters()).device
img_tensor_new = img_tensor_new.to(device)

print(f"✅ Preprocessed with CSRNetPreprocessor")
print(f"   Input image: {crowd_img.size[0]}x{crowd_img.size[1]}")
print(f"   Tensor shape: {img_tensor_new.shape}")
print(f"   Tensor device: {img_tensor_new.device}")
print(f"   Model device: {device}")
print(f"   Expected output: ({img_tensor_new.shape[2]//8}x{img_tensor_new.shape[3]//8})")

# Run inference with proper preprocessing
with torch.no_grad():
    density_map_new = model(img_tensor_new)
    count_new = density_map_new.sum().item()

print(f"\n📊 Results with proper preprocessing:")
print(f"   Raw count: {count_new:.2f}")
print(f"   Rounded: {int(round(count_new))}")
print(f"\n💡 Note: Dataset images without crowds may still produce low-confidence counts.")
print("   Try with a real crowd image for accurate counts!")

🖼️ Testing with dataset image using proper preprocessing...
✅ Preprocessed with CSRNetPreprocessor
   Input image: 483x360
   Tensor shape: torch.Size([1, 3, 360, 483])
   Tensor device: cuda:0
   Model device: cuda:0
   Expected output: (45x60)

📊 Results with proper preprocessing:
   Raw count: 623.12
   Rounded: 623

💡 Note: Dataset images without crowds may still produce low-confidence counts.
   Try with a real crowd image for accurate counts!

📊 Results with proper preprocessing:
   Raw count: 623.12
   Rounded: 623

💡 Note: Dataset images without crowds may still produce low-confidence counts.
   Try with a real crowd image for accurate counts!


## 🔍 DEBUG: Why is the count wrong?

Let's diagnose the checkpoint and model to understand why 1 person = 34

In [9]:
# Step 1: Check the checkpoint structure
print("🔍 CHECKPOINT DIAGNOSIS")
print("=" * 70)

checkpoint_path = '../../checkpoints/csrnet.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu')

print(f"\n📂 Checkpoint type: {type(checkpoint)}")

if isinstance(checkpoint, dict):
    print(f"\n🔑 Keys in checkpoint:")
    for key in checkpoint.keys():
        print(f"   - {key}")
    
    # Get state dict
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
        print("\n✅ Using 'state_dict' key")
    else:
        state_dict = checkpoint
        print("\n⚠️  Using checkpoint directly")
else:
    state_dict = checkpoint
    print("\n⚠️  Checkpoint is not a dict")

print(f"\n📊 Total parameters: {len(state_dict)}")
print(f"\n📋 First 10 layer names:")
for i, (key, value) in enumerate(list(state_dict.items())[:10]):
    print(f"   {i+1}. {key}: {value.shape}")

print(f"\n📋 Last 5 layer names:")
for i, (key, value) in enumerate(list(state_dict.items())[-5:]):
    print(f"   {i+1}. {key}: {value.shape}")

🔍 CHECKPOINT DIAGNOSIS

📂 Checkpoint type: <class 'dict'>

🔑 Keys in checkpoint:
   - state_dict
   - epoch
   - arch
   - optimizer
   - best_prec1

✅ Using 'state_dict' key

📊 Total parameters: 34

📋 First 10 layer names:
   1. frontend.0.weight: torch.Size([64, 3, 3, 3])
   2. frontend.0.bias: torch.Size([64])
   3. frontend.2.weight: torch.Size([64, 64, 3, 3])
   4. frontend.2.bias: torch.Size([64])
   5. frontend.5.weight: torch.Size([128, 64, 3, 3])
   6. frontend.5.bias: torch.Size([128])
   7. frontend.7.weight: torch.Size([128, 128, 3, 3])
   8. frontend.7.bias: torch.Size([128])
   9. frontend.10.weight: torch.Size([256, 128, 3, 3])
   10. frontend.10.bias: torch.Size([256])

📋 Last 5 layer names:
   1. backend.8.bias: torch.Size([128])
   2. backend.10.weight: torch.Size([64, 128, 3, 3])
   3. backend.10.bias: torch.Size([64])
   4. output_layer.weight: torch.Size([1, 64, 1, 1])
   5. output_layer.bias: torch.Size([1])


C:\Users\anush\AppData\Local\Temp\ipykernel_11836\3006355911.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')

In [10]:
# Step 2: Check what the model actually has loaded
print("🏗️  MODEL LOADED STATE")
print("=" * 70)

print(f"\nModel architecture:")
print(f"   Frontend: {len(list(model.frontend.parameters()))} params")
print(f"   Backend: {len(list(model.backend.parameters()))} params")
print(f"   Output: {len(list(model.output_layer.parameters()))} params")

print(f"\nFirst frontend layer:")
for name, param in list(model.frontend.named_parameters())[:1]:
    print(f"   {name}: {param.shape}")
    print(f"   First 10 values: {param.flatten()[:10].detach().cpu().numpy()}")

print(f"\nOutput layer:")
for name, param in model.output_layer.named_parameters():
    print(f"   {name}: {param.shape}")
    print(f"   First 10 values: {param.flatten()[:10].detach().cpu().numpy()}")

🏗️  MODEL LOADED STATE

Model architecture:
   Frontend: 20 params
   Backend: 12 params
   Output: 2 params

First frontend layer:
   0.weight: torch.Size([64, 3, 3, 3])
   First 10 values: [-0.5504041   0.14821805  0.53646326 -0.5774274   0.36223182  0.7716087
 -0.68409276 -0.04064278  0.48774746  0.17325266]

Output layer:
   weight: torch.Size([1, 64, 1, 1])
   First 10 values: [-0.05461554  0.01060466 -0.01452289 -0.00759119 -0.00712254  0.06178762
 -0.00139016  0.00212121 -0.00367885  0.05516844]
   bias: torch.Size([1])
   First 10 values: [0.0026974]


In [11]:
# Step 3: Check the density map output in detail
print("🧪 DENSITY MAP ANALYSIS")
print("=" * 70)

# Get the density map from earlier inference
print(f"\nDensity map statistics:")
print(f"   Shape: {density_map_new.shape}")
print(f"   Min value: {density_map_new.min().item():.6f}")
print(f"   Max value: {density_map_new.max().item():.6f}")
print(f"   Mean value: {density_map_new.mean().item():.6f}")
print(f"   Sum (count): {density_map_new.sum().item():.2f}")

print(f"\n📊 Density map value distribution:")
flat_density = density_map_new.flatten().detach().cpu().numpy()
print(f"   Positive values: {(flat_density > 0).sum()} / {len(flat_density)}")
print(f"   Negative values: {(flat_density < 0).sum()} / {len(flat_density)}")
print(f"   Zero values: {(flat_density == 0).sum()} / {len(flat_density)}")

print(f"\n🔢 First 20 density values:")
print(flat_density[:20])

print(f"\n⚠️  DIAGNOSIS:")
if density_map_new.mean().item() < 0:
    print("   ❌ Average density is NEGATIVE - Model not trained properly!")
elif density_map_new.sum().item() > 100:
    print("   ❌ Sum is way too high - Checkpoint may be wrong!")
elif abs(density_map_new.sum().item()) < 0.01:
    print("   ⚠️  Sum is near zero - Model might be undertrained")
else:
    print("   ✅ Density values look reasonable")

🧪 DENSITY MAP ANALYSIS

Density map statistics:
   Shape: torch.Size([1, 1, 45, 60])
   Min value: 0.000000
   Max value: 3.131865
   Mean value: 0.230786
   Sum (count): 623.12

📊 Density map value distribution:
   Positive values: 2624 / 2700
   Negative values: 0 / 2700
   Zero values: 76 / 2700

🔢 First 20 density values:
[0.         0.00062741 0.00062921 0.00162635 0.00079674 0.0024238
 0.0016056  0.00018063 0.         0.00335812 0.00254743 0.00383955
 0.0045463  0.00640569 0.00960663 0.00445848 0.00963224 0.03781974
 0.17454778 0.00979511]

⚠️  DIAGNOSIS:
   ❌ Sum is way too high - Checkpoint may be wrong!


## 💡 LIKELY CAUSES & SOLUTIONS

Based on the diagnosis above, here are the most likely issues:

In [12]:
print("""
🔴 PROBLEM: Getting count of 34 for 1 person

📋 POSSIBLE CAUSES:

1. ❌ WRONG CHECKPOINT
   - The csrnet.pth file may not be a properly trained CSRNet model
   - It might be from a different architecture or early training epoch
   - Solution: Get a proper pre-trained checkpoint from:
     * Original repo: https://github.com/leeyeehoo/CSRNet-pytorch
     * Or train your own on ShanghaiTech dataset

2. ❌ CHECKPOINT-ARCHITECTURE MISMATCH
   - The checkpoint may have been saved with DataParallel (module. prefix)
   - Or saved with different layer names than expected
   - Check the diagnosis above to see if layer names match

3. ❌ UNTRAINED OR POORLY TRAINED MODEL
   - If the checkpoint is from early training epochs, it won't work
   - Model needs 400+ epochs of training to converge
   - Random/untrained weights will produce garbage outputs

4. ❌ WRONG DATASET
   - If the model was trained on a different dataset (not ShanghaiTech)
   - It may produce wrong scales or outputs
   - CSRNet expects density maps with specific scaling

5. ⚠️  TEST IMAGE ISSUE
   - Random noise images will always produce wrong results
   - Need to test with REAL crowd images from ShanghaiTech
   - Single person in large empty space is hard for crowd counting models

📝 RECOMMENDED ACTIONS:

1. Download a pre-trained checkpoint:
   - From: https://drive.google.com/open?id=1QmB0KBnGR9q8_9-V-YG98G9fqBvBMy7u
   - This is the official Part A model
   
2. Or use a different checkpoint if you have one

3. Test with real crowd images, not random images or single person

4. Run the diagnostic cells above to see what's actually in your checkpoint
""")

print("\n🔍 Run the cells above to diagnose your specific checkpoint!")


🔴 PROBLEM: Getting count of 34 for 1 person

📋 POSSIBLE CAUSES:

1. ❌ WRONG CHECKPOINT
   - The csrnet.pth file may not be a properly trained CSRNet model
   - It might be from a different architecture or early training epoch
   - Solution: Get a proper pre-trained checkpoint from:
     * Original repo: https://github.com/leeyeehoo/CSRNet-pytorch
     * Or train your own on ShanghaiTech dataset

2. ❌ CHECKPOINT-ARCHITECTURE MISMATCH
   - The checkpoint may have been saved with DataParallel (module. prefix)
   - Or saved with different layer names than expected
   - Check the diagnosis above to see if layer names match

3. ❌ UNTRAINED OR POORLY TRAINED MODEL
   - If the checkpoint is from early training epochs, it won't work
   - Model needs 400+ epochs of training to converge
   - Random/untrained weights will produce garbage outputs

4. ❌ WRONG DATASET
   - If the model was trained on a different dataset (not ShanghaiTech)
   - It may produce wrong scales or outputs
   - CSRNet expec